# DeepSeek initial audio concepts
Generates one prompt family at a time and checkpoints after every class. CREMA-D intentionally has no `around` prompt because controlled acted-speech clips do not justify environmental concepts.

In [4]:
import os
from dotenv import load_dotenv
import data_utils
from concept_generation_deepseek import DeepSeekGenerator, generate_dataset_concepts, get_prompt

load_dotenv()
assert os.getenv('DEEPSEEK_API_KEY'), 'Create .env from .env.example and set DEEPSEEK_API_KEY'


In [5]:
dataset = 'cremad'  # esc50 | urbansound8k | cremad | audioset
prompt_type = 'important'  # CREMA-D: important | superclass; others also allow around
num_trials = 2
temperature = 0.4
model = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
classes = data_utils.get_dataset_classes(dataset)
save_path = f'data/concept_sets/deepseek_init/deepseek_{dataset}_{prompt_type}.json'
len(classes), classes[:10], model, save_path


(6,
 ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad'],
 'deepseek-v4-flash',
 'data/concept_sets/deepseek_init/deepseek_cremad_important.json')

In [6]:
# Inspect the exact prompt before spending API credit.
print(get_prompt(dataset, prompt_type, classes[0]))


Dataset: CREMA-D acted emotional speech. Target emotion: anger.
Generate 5 acoustic or prosodic properties of the speaker's voice that could help
recognize this emotion. Consider pitch, pitch variation, intensity, speech rate,
pauses, articulation, breathiness, roughness, vocal tension, rhythm, and spectral
quality. Use concrete phrases of 1-3 words. Do not use the emotion name or synonyms.
Do not generate situations, meanings, background sounds, bodily actions, music,
weather, facial expressions, or stereotypical semantic associations.


In [7]:
generator = DeepSeekGenerator(model=model)
generator.generate_concepts(get_prompt(dataset, prompt_type, classes[0]), temperature=0.2)


['high pitch',
 'fast speech rate',
 'loud intensity',
 'rough voice',
 'tense articulation']

In [8]:
results = generate_dataset_concepts(
    dataset=dataset, classes=classes, prompt_type=prompt_type,
    generator=generator, save_path=save_path, num_trials=num_trials,
    resume=True, temperature=temperature,
)
sum(map(len, results.values())), results[classes[0]], save_path


[1/6] generate anger
[2/6] generate disgust
[3/6] generate fear
[4/6] generate happy
[5/6] generate neutral
[6/6] generate sad


(45,
 ['high pitch',
  'loud intensity',
  'fast speech',
  'sharp articulation',
  'rough voice',
  'harsh voice',
  'low pitch variability'],
 'data/concept_sets/deepseek_init/deepseek_cremad_important.json')